# FLAG 2027 - Notebook 30: wide head search with an honest confirmation step

One head trains in ~20 s, so we can afford a real search instead of one change at a time. Risk: with 70 CV identities, the best of 36 random configs wins partly by luck. Guard: **stage A** searches on the usual folds with seeds 1-3; **stage B** re-runs the top 4 and the reference on DIFFERENT folds (new identity split) with NEW seeds 4-6. Only a config that wins in stage B counts.

Search space (all training-side, rules-safe): alignment loss (none / MSE w 0.3-3 / InfoNCE w 0.2-1), input dropout 0-0.9, within-identity re-pairing, head width 128/256/512, branch dropout 0.3-0.7, OPL weight 0.5-3, learning rate 5e-5 to 3e-4, identities per batch 16/32/64.

**How to run:** this notebook has ONE code cell. Paste it at the END of your NB28 notebook (same session, after NB28 Cell 3 finished), because it reuses NB28's features and harness. About 3.5 h. Resumable (re-run NB28 Cells 1-2 first after a disconnect). Paste the **SUMMARY**.

In [ ]:
# NB30 - wide head search (needs NB28 Cells 1-2 run in this session)
SUMMARY = []
class Branch2(nn.Module):
    def __init__(self, d_in, d, p):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(p))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP2(FOP):
    def __init__(self, n_class, d_face, d_voice, d=128, p=0.5):
        super().__init__(n_class, d_face, d_voice, d)
        self.face, self.voice = Branch2(d_face, d, p), Branch2(d_voice, d, p)
        for m in list(self.face.modules()) + list(self.voice.modules()):
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
def train_head2(F_raw, V_raw, all_tr, vsets, seed, cfg):
    align, w_al, tau = cfg.get('align'), cfg.get('w', 1.0), cfg.get('tau', 0.07)
    repair, p_in, d, p_br = cfg.get('repair', False), cfg.get('p_in', 0.0), cfg.get('d', 128), cfg.get('p_br', 0.5)
    alpha, lr, pid = cfg.get('alpha', ALPHA), cfg.get('lr', LR), cfg.get('pid', P_ID)
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP2(len(cl), XF.shape[1], XV.shape[1], d, p_br).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, MAX_EP, None, 0
    for ep in range(1, MAX_EP + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), pid, replace=False)
            j_np = np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen])
            jv_np = np.concatenate([g[rng.permutation(K_CLIP)] for g in j_np.reshape(pid, K_CLIP)]) if repair else j_np
            j, jv = torch.tensor(j_np, device=dev), torch.tensor(jv_np, device=dev)
            xf = (XF[tr[j]] - mf) / sf_; xv = (XV[tr[jv]] - mv) / sv
            if p_in > 0:
                xf = Fn.dropout(xf, p_in, True); xv = Fn.dropout(xv, p_in, True)
            fused, logits, ft, vt = model(xf, xv); lab = ytr[j]
            loss = Fn.cross_entropy(logits, lab) + alpha * opl_loss(fused, lab)
            if align == 'mse':
                loss = loss + w_al * ((ft - vt) ** 2).mean()
            elif align == 'nce':
                loss = loss + w_al * nce_loss(ft, vt, lab, tau)
            opt.zero_grad(); loss.backward(); opt.step()
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state); return model, best_ep, XF, XV
# ---- the search space: 36 random configs (fixed seed, so a rerun samples the same list) + the reference ----
def sample_cfg(r):
    a = r.choice(['none', 'mse', 'nce']); c = {}
    if a == 'mse':
        c.update(align='mse', w=float(r.choice([0.3, 1.0, 3.0])))
    elif a == 'nce':
        c.update(align='nce', w=float(r.choice([0.2, 0.5, 1.0])))
    c.update(p_in=float(r.choice([0.0, 0.3, 0.5, 0.7, 0.9])), repair=bool(r.random() < 0.5), d=int(r.choice([128, 256, 512])),
             p_br=float(r.choice([0.3, 0.5, 0.7])), alpha=float(r.choice([0.5, 1.0, 2.0, 3.0])), lr=float(r.choice([5e-5, 1e-4, 2e-4, 3e-4])), pid=int(r.choice([16, 32, 64])))
    return c
r_ = np.random.default_rng(2027); CFGS = {'S00_ref': {}}
for i in range(1, 37):
    CFGS['S%02d' % i] = sample_cfg(r_)
json.dump(CFGS, open(os.path.join(RESDIR, 'nb30_configs.json'), 'w'), indent=1)
# ---- stage B folds: a NEW identity split (seed 7) ----
rngB = np.random.default_rng(7); foldsB = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rngB.shuffle(members)
    for i, c in enumerate(members):
        foldsB[i % K].append(c)
FOLD_B = []
for k in range(K):
    val_ids = set(foldsB[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_B.append((tr_idx, [make_pairs(va_idx, seed=200 + k), make_pairs(va_idx, same_gender_neg=True, seed=300 + k)]))
CSV30 = os.path.join(RESDIR, 'nb30_cv.csv')
done30 = pd.read_csv(CSV30) if os.path.exists(CSV30) else pd.DataFrame(columns=['stage', 'combo', 'seed', 'fold']); t0 = time.time()
def run(stage, names, seeds, fold_data):
    global done30
    for name in names:
        for seed in seeds:
            for k in range(K):
                if ((done30.stage == stage) & (done30.combo == name) & (done30.seed == seed) & (done30.fold == k)).any():
                    continue
                tr_idx, vsets = fold_data[k]
                try:
                    model, best_ep, XF, XV = train_head2(F_RAW, V_RAW, np.concatenate([tr_idx, EXT]), vsets, seed, CFGS[name])
                    e_std, e_gen = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
                except Exception as e:
                    print(name, 'FAILED', repr(e)[:200]); best_ep, e_std, e_gen = -1, 50.0, 50.0
                row = pd.DataFrame([dict(stage=stage, combo=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
                row.to_csv(CSV30, mode='a', header=not os.path.exists(CSV30), index=False); done30 = pd.concat([done30, row])
            print('%s %-4s seed %d done | %.0f min' % (stage, name, seed, (time.time() - t0) / 60))
def tab(stage):
    R = pd.read_csv(CSV30); R = R[R.stage == stage].copy(); R['overall'] = (R['std'] + R['gender']) / 2
    return R.groupby(['combo', 'seed'])[['std', 'gender', 'overall']].mean().groupby('combo').mean().round(2).sort_values('overall')
run('A', list(CFGS), [1, 2, 3], FOLD_DATA)
TA = tab('A'); refA = TA.loc['S00_ref']
TOP = [c for c in TA.index if c != 'S00_ref'][:4]
run('B', ['S00_ref'] + TOP, [4, 5, 6], FOLD_B)
TB = tab('B'); refB = TB.loc['S00_ref']
note('STAGE A (search, usual folds, seeds 1-3) - top 8 of %d:' % len(TA)); note(TA.head(8).to_string())
note('reference in stage A: %.2f (std %.2f / gender %.2f)' % (refA['overall'], refA['std'], refA['gender']))
note('STAGE B (confirmation, NEW folds + NEW seeds 4-6):'); note(TB.to_string())
out = []
for c in TOP:
    dA, dB = TA.loc[c, 'overall'] - refA['overall'], TB.loc[c, 'overall'] - refB['overall']; dg = TB.loc[c, 'gender'] - refB['gender']; ds = TB.loc[c, 'std'] - refB['std']
    ok = dB <= -0.4 or (dg <= -0.5 and ds <= 0.2)
    out.append('%s %s: A %+.2f | B %+.2f (std %+.2f, gender %+.2f) -> %s' % (c, json.dumps(CFGS[c]), dA, dB, ds, dg, 'ADOPT' if ok else 'reject'))
note('VERDICTS (count only stage B: overall <= -0.4, or gender <= -0.5 with std <= +0.2):')
for o in out:
    note(o)
print('=' * 70); print('SUMMARY  (copy everything between the lines and paste to Claude)'); print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)